# All you need to modify

In [32]:
import pandas as pd
import os
import numpy as np


students_sheets = pd.read_excel("/content/Students_CSD3_Weekly Reports (Guo) 0504 (Copy) - May 6 2026.xlsx", sheet_name=None)

adults_sheets = "/content/Adults_CSD3_Weekly Reports (Guo) 0504 (Copy) - May 6 2026.xlsx"

all_sheets = "/content/All_CSD3_Weekly Reports (Guo) 0504 (Copy) - May 6 2026.xlsx"

# Student Summary Statistics - Target # of Students Served (don't include Total)
target_values = [152,200,100]




/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Code Part (don't need to modify)

## 🍀Student Summary Statistics:


In [49]:
df_part_by_hour=  students_sheets['Participants By Hour Band (Site']
df_part_by_hour.head()

1,Institution,Site,Registered Adults,Attended Adults,Registered Youth,Attended Youth,0 Hours,Less Than 15 Hours,15-44 Hours,45-89 Hours,90-179 Hours,180-269 Hours,270+ Hours,15+ Hours,Last Attendance Date
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Institution,Site,Registered Adults,Attended Adults,Registered Youth,Attended Youth,0 Hours,Less Than 15 Hours,15-44 Hours,45-89 Hours,90-179 Hours,180-269 Hours,270+ Hours,15+ Hours,Last Attendance Date
2,"Community School District 3, NYCDOE 8026",High School of Arts and Technology,65,65,360,317,43,212,88,14,3,0,0,105,2026-04-23 00:00:00
3,NaN,High School of Arts and Technology,65,65,360,317,43,212,88,14,3,0,0,105,NaN
4,"Community School District 3, NYCDOE 8026",P.S. 165 Robert E. Simon,168,165,439,438,1,208,53,22,139,16,0,230,2026-05-05 00:00:00


In [50]:
# Use the 3rd row (index 2) as column names
df_part_by_hour.columns = df_part_by_hour.iloc[1]

# Drop the first three rows and reset index
df_part_by_hour = df_part_by_hour.iloc[2:].reset_index(drop=True)

df_part_by_hour.head(10)

1,Institution,Site,Registered Adults,Attended Adults,Registered Youth,Attended Youth,0 Hours,Less Than 15 Hours,15-44 Hours,45-89 Hours,90-179 Hours,180-269 Hours,270+ Hours,15+ Hours,Last Attendance Date
0,"Community School District 3, NYCDOE 8026",High School of Arts and Technology,65,65,360,317,43,212,88,14,3,0,0,105,2026-04-23 00:00:00
1,NaN,High School of Arts and Technology,65,65,360,317,43,212,88,14,3,0,0,105,NaN
2,"Community School District 3, NYCDOE 8026",P.S. 165 Robert E. Simon,168,165,439,438,1,208,53,22,139,16,0,230,2026-05-05 00:00:00
3,NaN,P.S. 165 Robert E. Simon,168,165,439,438,1,208,53,22,139,16,0,230,NaN
4,"Community School District 3, NYCDOE 8026",STEM Institute,2,0,145,145,0,40,38,42,25,0,0,105,2026-04-03 00:00:00
5,NaN,STEM Institute,2,0,145,145,0,40,38,42,25,0,0,105,NaN
6,NaN,NaN,235,230,944,900,44,460,179,78,167,16,0,440,NaN


In [51]:
rows_to_drop = []

for idx, row in df_part_by_hour.iterrows():
    institution = row['Institution']
    site = row['Site']

    is_empty_institution = pd.isna(institution) or str(institution).strip() == "NaN"
    is_empty_site = pd.isna(site) or str(site).strip() == "NaN"

    if (is_empty_institution and is_empty_site) or not is_empty_institution:
        rows_to_drop.append(idx)

df_part_by_hour = df_part_by_hour.drop(index=rows_to_drop).reset_index(drop=True)
df_part_by_hour.head(10)

1,Institution,Site,Registered Adults,Attended Adults,Registered Youth,Attended Youth,0 Hours,Less Than 15 Hours,15-44 Hours,45-89 Hours,90-179 Hours,180-269 Hours,270+ Hours,15+ Hours,Last Attendance Date
0,NaN,High School of Arts and Technology,65,65,360,317,43,212,88,14,3,0,0,105,NaN
1,NaN,P.S. 165 Robert E. Simon,168,165,439,438,1,208,53,22,139,16,0,230,NaN
2,NaN,STEM Institute,2,0,145,145,0,40,38,42,25,0,0,105,NaN


In [52]:
# the average column from Daily Site Attendance Summary
daily_site_att = students_sheets['Daily Site Attendance Summary']

daily_site_att.columns = daily_site_att.iloc[2]  # Set 3rd row as header
daily_site_att = daily_site_att.iloc[3:]          # Drop first 3 rows
daily_site_att.columns.name = None                # Remove column index name
daily_site_att = daily_site_att.reset_index(drop=True)

daily_site_att = daily_site_att[['Total']]        # Keep only Total column

daily_site_att = daily_site_att.iloc[:-1]         # Drop last row
daily_site_att['Total'] = daily_site_att['Total'].str.extract(r'(\d+\.?\d*)')  # Keep number only


In [53]:
print(df_part_by_hour.columns)

Index(['Institution', 'Site', 'Registered Adults', 'Attended Adults',
       'Registered Youth', 'Attended Youth', '0 Hours', 'Less Than 15 Hours',
       '15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours',
       '270+ Hours', '15+ Hours', 'Last Attendance Date'],
      dtype='object', name=1)


In [54]:
all_cols = ['0 Hours', 'Less Than 15 Hours', '15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours', '270+ Hours']
served_cols = ['Less Than 15 Hours', '15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours', '270+ Hours']
plus15_cols = ['15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours', '270+ Hours']
plus90_cols = ['90-179 Hours', '180-269 Hours', '270+ Hours']

# Filter each col list to only include columns that exist in the dataframe
existing_cols = df_part_by_hour.columns.tolist()
all_cols     = [c for c in all_cols     if c in existing_cols]
served_cols  = [c for c in served_cols  if c in existing_cols]
plus15_cols  = [c for c in plus15_cols  if c in existing_cols]
plus90_cols  = [c for c in plus90_cols  if c in existing_cols]

# Convert to numeric
df_part_by_hour[all_cols] = df_part_by_hour[all_cols].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)

# Build completely new dataframe
df_totals = pd.DataFrame({
    '[Total # Enrolled]': df_part_by_hour[all_cols].sum(axis=1),
    '[Total # Served]':   df_part_by_hour[served_cols].sum(axis=1),
    '[Total # 15+]':      df_part_by_hour[plus15_cols].sum(axis=1),
    '[Total # 90+]':      df_part_by_hour[plus90_cols].sum(axis=1)
})

# Insert as the first column (position 0)
df_totals.insert(0, '[Target # of students served]', target_values)

In [55]:
df_totals.head(10)

,[Target # of students served],[Total # Enrolled],[Total # Served],[Total # 15+],[Total # 90+]
0,152,360,317,105,3
1,200,439,438,230,155
2,100,145,145,105,25


In [56]:
## the average column from Daily Site Attendance Summary
daily_site_att['Total'] = daily_site_att['Total'].astype(int)
daily_site_att = daily_site_att.rename(columns={'Total': 'Avg. # of Students Per Day'})
# Insert after [Total # Served] (now at position 2)
df_totals.insert(3, 'Avg. # of Students Per Day', daily_site_att['Avg. # of Students Per Day'].values)
print(df_totals)

   [Target # of students served]  [Total # Enrolled]  [Total # Served]  \
0                            152                 360               317   
1                            200                 439               438   
2                            100                 145               145   

   Avg. # of Students Per Day  [Total # 15+]  [Total # 90+]  
0                          30            105              3  
1                         101            230            155  
2                          44            105             25  


In [57]:
##INSERT SCHOOL NAMES
# Create an ordered list of valid school names, skipping NaN and 'Subtotal'
school_names = [
    row['Site']
    for _, row in df_part_by_hour.iterrows()
    if pd.notna(row['Site']) and row['Site'] != 'Subtotal' and row['Site'] != 'Total'
]

school_names

# Add the actual Site names as the leftmost column
df_totals.insert(0, 'School', school_names)

# Create the final total row
total_row = pd.DataFrame(df_totals.iloc[:, 1:].sum()).T
total_row.insert(0, 'School', 'Total')

# Append the total row
df_totals = pd.concat([df_totals, total_row], ignore_index=True)

df_totals

,School,[Target # of students served],[Total # Enrolled],[Total # Served],Avg. # of Students Per Day,[Total # 15+],[Total # 90+]
0,High School of Arts and Technology,152,360,317,30,105,3
1,P.S. 165 Robert E. Simon,200,439,438,101,230,155
2,STEM Institute,100,145,145,44,105,25
3,Total,452,944,900,175,440,183


In [58]:
# Create formatted display columns directly (no helper % columns)

df_totals['# of students 15+ hrs total (% of Target)'] = (
    df_totals['[Total # 15+]'].astype(int).astype(str) +
    " (" +
    (
        (df_totals['[Total # 15+]'] /
         df_totals['[Target # of students served]']) * 100
    ).round().astype(int).astype(str) +
    "%)"
)

df_totals['# of students 90+ hrs total (% of Target)'] = (
    df_totals['[Total # 90+]'].astype(int).astype(str) +
    " (" +
    (
        (df_totals['[Total # 90+]'] /
         df_totals['[Target # of students served]']) * 100
    ).round().astype(int).astype(str) +
    "%)"
)
df_totals = df_totals.drop(columns=['[Total # 15+]', '[Total # 90+]'])
df_totals = df_totals.reset_index(drop=True)
df_totals


,School,[Target # of students served],[Total # Enrolled],[Total # Served],Avg. # of Students Per Day,# of students 15+ hrs total (% of Target),# of students 90+ hrs total (% of Target)
0,High School of Arts and Technology,152,360,317,30,105 (69%),3 (2%)
1,P.S. 165 Robert E. Simon,200,439,438,101,230 (115%),155 (78%)
2,STEM Institute,100,145,145,44,105 (105%),25 (25%)
3,Total,452,944,900,175,440 (97%),183 (40%)


In [59]:
# --- ADD BLOCK 1 HERE: define the styler ---
def color_pct_cols(val):
    try:
        pct = int(val.split('(')[1].replace('%)', '').strip())
        return 'color: green' if pct >= 100 else 'color: red'
    except:
        return ''

styled_totals = df_totals.style.applymap(
    color_pct_cols,
    subset=['# of students 15+ hrs total (% of Target)']
)



/tmp/ipykernel_8165/3588560171.py:9: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_totals = df_totals.style.applymap(


##🌷Family Component Summary Statistics (Last Column)

In [60]:

# Load the specific sheet for Attendance Hours, skipping the top 2 rows
df_hours = pd.read_excel(adults_sheets, sheet_name="Participant Attendance Hours", skiprows=2)

# Ensure "HoursPresent" is numeric
df_hours["HoursPresent"] = pd.to_numeric(df_hours["HoursPresent"], errors='coerce')

# Clean the ParticipantId: convert to string and remove any trailing '.0' in case pandas loaded it as a float
df_hours["ParticipantId"] = df_hours["ParticipantId"].astype(str).str.replace(r'\.0$', '', regex=True)

# # Filter for rows where HoursPresent is greater than 0
# df_active = df_hours[df_hours["HoursPresent"] > 0]

# Filter for rows where HoursPresent is > 0 AND the ParticipantId length is NOT exactly 9 digits
df_active = df_hours[(df_hours["HoursPresent"] > 0) & (df_hours["ParticipantId"].str.len() != 9)]

# Group by 'Site' and count unique adults using 'ParticipantId'
result = df_active.groupby("Site")["ParticipantId"].nunique().reset_index()

# Rename the column for better presentation
result.rename(columns={"ParticipantId": "Parents Served (Total)"}, inplace=True)

# Calculate the total sum of the 'Parents Served (Total)' column
total_parents = result["Parents Served (Total)"].sum()

# Append the total row to the dataframe (using loc is the cleanest way)
result.loc[len(result)] = {"Site": "Total", "Parents Served (Total)": total_parents}

# Display the final table
display(result)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Site,Parents Served (Total)
0,High School of Arts and Technology,65
1,P.S. 165 Robert E. Simon,165
2,Total,230


##  🌸Participant Demographics - missing info

In [61]:
df_part_demo=  students_sheets['Participant Demographics']
df_part_demo.head()

2,Institution,Site,Last Name,First Name,ParticipantID,State ParticipantID,Adult,Date Of Birth,School,Grade Level,Gender,Race/Ethnicity,English Learner Status,Lunch Status,Special Education Status,IDEA Disability Type
0,Participant Demographics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Institution,Site,Last Name,First Name,ParticipantID,State ParticipantID,Adult,Date Of Birth,School,Grade Level,Gender,Race/Ethnicity,English Learner Status,Lunch Status,Special Education Status,IDEA Disability Type
3,"Community School District 3, NYCDOE 8026",P.S. 165 Robert E. Simon,Aafjes,Lucas,250138685,250138685,No,2020-05-08 00:00:00,P.S. 165 Robert E. Simon,KG,Male,Not Entered,No,Not Entered,No,Not Entered
4,"Community School District 3, NYCDOE 8026",High School of Arts and Technology,Abad Hernandez,Isbeth,232761528,2648790956,No,2008-04-18 00:00:00,High School of Arts and Technology,11,Female,Race and Ethnicity Unknown,No,Full price,No,Not Entered


In [62]:
# Use the 3rd row (index 2) as column names
df_part_demo.columns = df_part_demo.iloc[2]

# Drop the first three rows and reset index
df_part_demo = df_part_demo.iloc[3:].reset_index(drop=True)

df_part_demo.head()
print(df_part_demo.dtypes)

2
Institution                 object
Site                        object
Last Name                   object
First Name                  object
ParticipantID               object
State ParticipantID         object
Adult                       object
Date Of Birth               object
School                      object
Grade Level                 object
Gender                      object
Race/Ethnicity              object
English Learner Status      object
Lunch Status                object
Special Education Status    object
IDEA Disability Type        object
dtype: object


In [63]:
def summarize_missing_by_school(df, columns_to_check, category_col="Site"):
    if category_col not in df.columns:
        raise ValueError(f"{category_col} not found in DataFrame columns")

    missing_site_rows = df[df[category_col].isna() | (df[category_col].astype(str).str.strip() == "")].copy()

    subset = df[columns_to_check + [category_col]].copy()
    subset_for_grouping = subset[subset[category_col].notna()].copy()
    subset_for_grouping[category_col] = (
        subset_for_grouping[category_col].astype(str).str.title()
    )

    ## Flagging missing values per col
    for col in columns_to_check:
        cleaned = subset_for_grouping[col].astype(str).str.strip() #here it removed teh whitespace and convert to string for consistent processing
        not_entered_mask = cleaned.str.lower() == "not entered" #flag any cells that say "Not Entered"
        if col == "Gender":
            valid_gender = cleaned.str.title().isin(["Male", "Female", "Non-Binary"])
            subset_for_grouping[col + "_missing"] = ((~valid_gender) | not_entered_mask).astype(int)
        else:
            subset_for_grouping[col + "_missing"] = (
                subset_for_grouping[col].isna() | not_entered_mask
            ).astype(int)
## Validating Participant IDs
    pid  = df.loc[subset_for_grouping.index, "ParticipantID"].astype(str).str.strip()
    spid = df.loc[subset_for_grouping.index, "State ParticipantID"].astype(str).str.strip()
    valid_pid  = pid.str.match(r'^[12]\d{8}$') # Assuming valid IDs start with 1 or 2 followed by 8 digits
    valid_spid = spid.str.match(r'^\d{10}$')# 10 digits
    subset_for_grouping["ParticipantID_missing"]       = (~valid_pid).astype(int)
    subset_for_grouping["State ParticipantID_missing"] = (~valid_spid).astype(int)

    # valid_osis = (pid == spid) & valid_pid & valid_spid
    # subset_for_grouping["OSIS_missing"] = (~valid_osis).astype(int) #we are not doing this part anymore

    missing_cols = (
        [col + "_missing" for col in columns_to_check]
        + ["ParticipantID_missing", "State ParticipantID_missing"]
    )
    pivot = (
        subset_for_grouping
        .groupby(category_col)[missing_cols]
        .sum()
        .reset_index()
    )
    total_row = pd.DataFrame(pivot[missing_cols].sum()).T
    total_row[category_col] = "Total"
    pivot = pd.concat([pivot, total_row], ignore_index=True)

    all_missing_flags = df.copy()
    pid_all  = all_missing_flags["ParticipantID"].astype(str).str.strip()
    spid_all = all_missing_flags["State ParticipantID"].astype(str).str.strip()
    valid_pid_all  = pid_all.str.match(r'^[12]\d{8}$')
    valid_spid_all = spid_all.str.match(r'^\d{10}$')
    # valid_osis_all = (pid_all == spid_all) & valid_pid_all & valid_spid_all

    for col in columns_to_check:
        cleaned = all_missing_flags[col].astype(str).str.strip()
        not_entered_mask = cleaned.str.lower() == "not entered"
        if col == "Gender":
            valid_gender = cleaned.str.title().isin(["Male", "Female", "Non-Binary"])
            all_missing_flags[col + "_missing"] = ((~valid_gender) | not_entered_mask).astype(int)
        else:
            all_missing_flags[col + "_missing"] = (all_missing_flags[col].isna() | not_entered_mask).astype(int)

    all_missing_flags["ParticipantID_missing"]       = (~valid_pid_all).astype(int)
    all_missing_flags["State ParticipantID_missing"] = (~valid_spid_all).astype(int)
    # all_missing_flags["OSIS_missing"]                = (~valid_osis_all).astype(int)

    # # Flag DOBs that are out of expected range: born after 2023 (too young) or before 2004 (too old)
    # ✅ Works with datetime objects directly
    dob_parsed = pd.to_datetime(all_missing_flags["Date Of Birth"], errors="coerce")
    all_missing_flags["DOB_too_young"] = (
        (dob_parsed.dt.year > 2023) | (dob_parsed.dt.year < 2004)
    ).astype(int)


#flagging and pulling rows with any missing/suspicious values
    flag_cols = (
        [col + "_missing" for col in columns_to_check]
        + ["ParticipantID_missing",  "DOB_too_young"] #"State ParticipantID_missing",
    )
    total_missing_rows = all_missing_flags[all_missing_flags[flag_cols].sum(axis=1) > 0].copy()

    # ← NEW: also pull rows where DOB is suspiciously young, even if nothing else is missing
    young_dob_rows = all_missing_flags[all_missing_flags["DOB_too_young"] == 1].copy()  # ← NEW

    # ADD DOB young counts to the pivot table
    dob_young_counts = all_missing_flags.groupby(category_col)["DOB_too_young"].sum()
    pivot = pivot.set_index("Site")
    pivot["Date Of Birth_missing"] += pivot.index.map(dob_young_counts).fillna(0).astype(int)
    pivot = pivot.reset_index()
    # rearrange the columns and rename some for better presentation
    pivot = pivot.rename(columns={
    "Date Of Birth_missing": "DOB_missing",
    "State ParticipantID_missing": "10digit_State ParticipantID_missing"
    })[["Site", "DOB_missing", "ParticipantID_missing", "Grade Level_missing", "Gender_missing", "10digit_State ParticipantID_missing"]]



    return pivot, missing_site_rows, total_missing_rows, flag_cols, young_dob_rows

- apply highlights

In [64]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

def apply_missing_highlights(
    output_filename,
    total_missing_rows,
    flag_cols,
    young_dob_rows,
    staff_missing_rows=None,
    staff_flag_cols=None,
    site_tables=None
):
    red_fill  = PatternFill("solid", start_color="FF9999", end_color="FF9999")
    blue_fill = PatternFill("solid", start_color="9999FF", end_color="9999FF")

    flag_to_original = {
        fc: fc[: -len("_missing")]
        for fc in flag_cols
        if fc.endswith("_missing") and fc != "State ParticipantID_missing"
    }

    wb = load_workbook(output_filename)

    # --- RED HIGHLIGHTS: student missing info, per "Missing - {site}" sheet ---
    for site, group in total_missing_rows.groupby("Site"):
        sheet_name = ("Missing - " + str(site))[:31]   # must match how sheets were named

        if sheet_name not in wb.sheetnames:
            print(f"Sheet not found, skipping: {sheet_name}")
            continue

        ws = wb[sheet_name]
        header = {cell.value: cell.column for cell in ws[1]}

        for row_idx, (_, row) in enumerate(group.iterrows(), start=2):
            for flag_col, orig_col in flag_to_original.items():
                if orig_col in header and row.get(flag_col, 0) == 1:
                    ws.cell(row=row_idx, column=header[orig_col]).fill = red_fill

    # --- RED HIGHLIGHTS: staff missing info pull-out sheet ---
    if staff_missing_rows is not None and staff_flag_cols is not None and "Pull out - Missing Staff Info" in wb.sheetnames:
        ws_staff = wb["Pull out - Missing Staff Info"]
        staff_header = {cell.value: cell.column for cell in ws_staff[1]}

        # Map each flag column to the original column shown in the pull-out sheet.
        staff_flag_to_original = {
            "Staff Type_missing": "Staff Type",
            "Employment Type_missing": "Compensation Type",
            "Funded by 21st CCLC_missing": "Funder",
        }

        for row_idx, (_, row) in enumerate(staff_missing_rows.iterrows(), start=2):
            for flag_col in staff_flag_cols:
                orig_col = staff_flag_to_original.get(flag_col, flag_col.replace("_missing", ""))
                if orig_col in staff_header and row.get(flag_col, 0) == 1:
                    ws_staff.cell(row=row_idx, column=staff_header[orig_col]).fill = red_fill

    # --- BLUE HIGHLIGHTS: Young DOB sheet ---
    if "Pull out - Young DOB" in wb.sheetnames:
        ws2 = wb["Pull out - Young DOB"]
        header2 = {cell.value: cell.column for cell in ws2[1]}

        if "Date Of Birth" in header2:
            dob_col_idx = header2["Date Of Birth"]
            for row_idx in range(2, len(young_dob_rows) + 2):
                ws2.cell(row=row_idx, column=dob_col_idx).fill = blue_fill

    # --- RED HIGHLIGHTS: missing cells in each site summary report ---
    if site_tables is not None:
        for site_name, final_df in site_tables.items():
            safe_sheet_name = (
                str(site_name)[:31]
                .replace(":", "").replace("/", "").replace("\\", "")
                .replace("?", "").replace("*", "")
            )

            if safe_sheet_name not in wb.sheetnames:
                print(f"Site summary sheet not found, skipping: {safe_sheet_name}")
                continue

            ws_site = wb[safe_sheet_name]
            header_site = {cell.value: cell.column for cell in ws_site[1]}

            for row_idx in range(2, ws_site.max_row + 1):
                for col_idx in range(1, ws_site.max_column + 1):
                    cell = ws_site.cell(row=row_idx, column=col_idx)

                    if cell.value is None or str(cell.value).strip() in ["", "-"]:
                        cell.fill = red_fill

    wb.save(output_filename)
    print("Highlights applied.")



 call the function here

In [65]:


columns_of_interest = ["Date Of Birth", "Grade Level", "Gender"]

missing_summary, missing_site_rows, total_missing_rows, flag_cols, young_dob_rows = summarize_missing_by_school(
    df_part_demo, columns_of_interest
)


In [66]:
# Staff Details - missing info summary
# This uses the "Staff Details" report directly.

df_staff = pd.read_excel(all_sheets, sheet_name="Staff Details", skiprows=2)

def summarize_staff_missing_info(df_staff, site_col="Site"):
    df = df_staff.copy()

    # Keep rows with a valid site
    df = df[df[site_col].notna() & (df[site_col].astype(str).str.strip() != "")].copy()

    # Make sure key columns exist
    for col in ["Email Address", "First Name", "Last Name", "Staff Type", "Compensation Type", "Funder"]:
        if col not in df.columns:
            df[col] = ""

    # Create a unique staff ID. Email is best; if email is blank, use first + last name.
    email = df["Email Address"].astype(str).str.strip().str.lower()
    name_id = (
        df["First Name"].astype(str).str.strip().str.lower()
        + "|"
        + df["Last Name"].astype(str).str.strip().str.lower()
    )
    df["_staff_id"] = np.where(
        email.ne("") & email.ne("nan"),
        email,
        name_id
    )

    # Count each staff member once per site
    df = df.drop_duplicates(subset=[site_col, "_staff_id"]).copy()

    staff_type = df["Staff Type"].astype(str).str.strip()
    comp_type = df["Compensation Type"].astype(str).str.strip()
    funder = df["Funder"].astype(str).str.strip()

    # Staff Type: blank / Not Entered / Other are not accepted
    df["Staff Type_missing"] = (
        staff_type.eq("")
        | staff_type.str.lower().isin(["nan", "not entered", "other"])
    ).astype(int)

    # Compensation Type: must be Paid or Volunteer
    df["Employment Type_missing"] = (
        comp_type.eq("")
        | comp_type.str.lower().isin(["nan", "not entered"])
        | ~comp_type.str.lower().isin(["paid", "volunteer"])
    ).astype(int)

    # Funder:
    # - Paid staff must have a funder and it should be 21st CCLC.
    # - Volunteers may leave Funder blank.
    is_volunteer = comp_type.str.lower().eq("volunteer")
    funder_blank = funder.eq("") | funder.str.lower().isin(["nan", "not entered"])
    funder_not_21cclc = ~funder.str.contains("21", case=False, na=False)

    df["Funded by 21st CCLC_missing"] = (
        ((~is_volunteer) & funder_blank)
        | ((~funder_blank) & funder_not_21cclc)
    ).astype(int)

    staff_missing_summary = (
        df.groupby(site_col)
        .agg(
            **{
                "# of Active Program Staff": ("_staff_id", "nunique"),
                "Staff Type": ("Staff Type_missing", "sum"),
                "Employment Type": ("Employment Type_missing", "sum"),
                "Funded by 21st CCLC": ("Funded by 21st CCLC_missing", "sum"),
            }
        )
        .reset_index()
        .rename(columns={site_col: "Site"})
    )

    total_row = {"Site": "Total"}
    for col in staff_missing_summary.columns:
        if col != "Site":
            total_row[col] = staff_missing_summary[col].sum()

    staff_missing_summary = pd.concat(
        [staff_missing_summary, pd.DataFrame([total_row])],
        ignore_index=True
    )

    staff_flag_cols = [
        "Staff Type_missing",
        "Employment Type_missing",
        "Funded by 21st CCLC_missing",
    ]

    staff_missing_rows = df[df[staff_flag_cols].sum(axis=1) > 0].copy()

    return staff_missing_summary, staff_missing_rows, staff_flag_cols

staff_missing_summary, staff_missing_rows, staff_flag_cols = summarize_staff_missing_info(df_staff)

display(staff_missing_summary)


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Site,# of Active Program Staff,Staff Type,Employment Type,Funded by 21st CCLC
0,High School of Arts and Technology,42,0,0,0
1,P.S. 165 Robert E. Simon,41,20,0,0
2,STEM Institute,24,2,0,0
3,Total,107,22,0,0


## 🪻Site Summary Report

In [67]:
import pandas as pd
import numpy as np
import os



# Load the specific sheets and skip rows accordingly
df_act = pd.read_excel(all_sheets, sheet_name="Activity-Session Details", skiprows=2)
df_enr = pd.read_excel(all_sheets, sheet_name="Session Enrollment by Session", skiprows=2)
# Excel limits sheet names to 31 chars, so 'Summary' is cut off to 'Summa'
df_att = pd.read_excel(all_sheets, sheet_name="Daily Activity Attendance Summa", skiprows=4)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [68]:
# 2. Extract and format the Activity-Session Details
cols_act = ["Site", "Activity", "Session", "Days Scheduled", "Session Start Date"]
df_act = df_act[cols_act].copy()
# Convert "Days Scheduled" to numeric, converting errors/blanks to NaN
df_act["Days Scheduled"] = pd.to_numeric(df_act["Days Scheduled"], errors='coerce')
df_act["Session Start Date"] = pd.to_datetime(df_act["Session Start Date"], errors='coerce')
today = pd.Timestamp.today().normalize()

In [69]:
# 3. Extract and format Session Enrollment by Session
cols_enr = ["Site", "Activity", "Session", "Enrolled Count"]
df_enr = df_enr[cols_enr].copy()
# Convert to numeric and rename the column
df_enr["Enrolled Count"] = pd.to_numeric(df_enr["Enrolled Count"], errors='coerce')
df_enr.rename(columns={"Enrolled Count": "Enrolled Participant"}, inplace=True)

In [70]:
# 4. Extract and format Daily Activity Attendance Summary
cols_att = ["Site", "Activity", "Session", "Total"]
df_att = df_att[cols_att].copy()

# Function to extract the numeric value from the "Total" column string
def extract_average(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace('Average:', '').strip()
    try:
        return float(val_str)
    except ValueError:
        return np.nan
# load the numeric and rename column
df_att["Total"] = df_att["Total"].apply(extract_average).round(0)
df_att.rename(columns={"Total": "Average Daily Attendance"}, inplace=True)

In [71]:
# 5. Get a list of unique, valid sites
sites = df_act['Site'].dropna().unique().tolist()
# Filter out any aggregate rows like 'Total' or empty strings
sites = [s for s in sites if str(s).strip() != '' and not str(s).startswith('Total') and not str(s).startswith('Average')]

# 6. Create a dictionary to hold the merged DataFrame for each site
site_tables = {}

for site in sites:
    # Filter the three dataframes for the current site
    site_act = df_act[df_act['Site'] == site].copy()
    site_enr = df_enr[df_enr['Site'] == site].copy()
    site_att = df_att[df_att['Site'] == site].copy()

    # Merge Activity and Enrollment
    merged_site = pd.merge(site_act, site_enr, on=["Site", "Activity", "Session"], how="outer")

    # Merge the result with Attendance
    merged_site = pd.merge(merged_site, site_att, on=["Site", "Activity", "Session"], how="outer")

    merged_site = merged_site[~(merged_site['Session Start Date'] >= today)]

    # Delete the 'Session Start Date' column from the final table
    if 'Session Start Date' in merged_site.columns:
        merged_site = merged_site.drop(columns=['Session Start Date'])

    # Replace NaN (null) values with "-"
    merged_site = merged_site.fillna("-")

    # merged_site = merged_site.replace(r'^\s*$', '-', regex=True)

    # Store in our dictionary
    site_tables[site] = merged_site.sort_values(by=['Session'], ascending=True).reset_index(drop=True)

In [72]:
# 7. Print or preview the results
for site_name, final_df in site_tables.items():
    print(f"\n{'='*50}")
    print(f"Site: {site_name}")
    print(f"{'='*50}")
    # Display the first few rows for validation in Colab
    display(final_df.head())


Site: High School of Arts and Technology


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,High School of Arts and Technology,College and Career Readiness,AMP NBC,16,15,13.0
1,High School of Arts and Technology,Academic Enrichment,AP Psych Test Prep,2,6,6.0
2,High School of Arts and Technology,Academic Enrichment,AP Seminar Test Prep,5,23,9.0
3,High School of Arts and Technology,Arts & Music,Anime Club,24,79,10.0
4,High School of Arts and Technology,Academic Enrichment,Book Club,6,18,9.0



Site: P.S. 165 Robert E. Simon


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,P.S. 165 Robert E. Simon,"Sports, Fitness, & Wellness",African Dance/Creative Movement | Cycle 1 | Mo...,11.0,17.0,15.0
1,P.S. 165 Robert E. Simon,"Sports, Fitness, & Wellness",African Dance/Creative Movement | Cycle 2 | Th...,8.0,15.0,14.0
2,P.S. 165 Robert E. Simon,Arts & Music,African Drumming | Cycle 1 | Monday | K-2,11.0,15.0,14.0
3,P.S. 165 Robert E. Simon,Arts & Music,African Drumming | Cycle 2| Thursday | K-2,8.0,19.0,18.0
4,P.S. 165 Robert E. Simon,Arts & Music,Ballet Hispanico | Friday | 3-5,8.0,23.0,17.0



Site: STEM Institute


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,STEM Institute,Arts & Music,African Dance,68,13.0,12.0
1,STEM Institute,Arts & Music,Drumming,61,12.0,12.0
2,STEM Institute,Academic Enrichment,ELA/Math Tutoring (Mr. Nesbitt),49,14.0,14.0
3,STEM Institute,Academic Enrichment,ELA/Math Tutoring (Ms. Cabrera),50,13.0,13.0
4,STEM Institute,Academic Enrichment,ELA/Math Tutoring (Ms. Gil),51,13.0,14.0





##🌈Full Report Download

In [73]:
# from google.colab import files
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

output_filename = "Processed_Site_Tables.xlsx"

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:

    styled_totals.to_excel(writer, sheet_name="Student Summary Statistics", index=False)
    print(f"Successfully created sheet for: Student Summary Statistics (Rows: {len(df_totals)})")

    # Write student missing summary and staff missing summary into the same shee
    missing_summary.to_excel(writer, sheet_name="Missing Student Summary", index=False)
    print(f"Successfully created sheet for: Missing Student Summary (Rows: {len(missing_summary)})")


    missing_site_rows.to_excel(writer, sheet_name="Pull out - Missing Site Info", index=False)
    print(f"Successfully created sheet for: Missing Site Info (Rows: {len(missing_site_rows)})")

    # NEW: one tab per school#remove cols not needed
    hide_cols = {'Race/Ethnicity', 'English Learner Status', 'Lunch Status', 'Special Education Status', 'IDEA Disability Type'}
    display_cols = [
    c for c in total_missing_rows.columns
    if not c.endswith("_missing")
    and c != "DOB_too_young"
    and c not in hide_cols]#remove cols not needed

    for site, group in total_missing_rows.groupby("Site"):
        safe_name = ("Missing - " + str(site))[:31]
        group[display_cols].to_excel(writer, sheet_name=safe_name, index=False)
        print(f"Successfully created sheet for: {safe_name} (Rows: {len(group)})")




    young_display_cols = [c for c in young_dob_rows.columns if not c.endswith("_missing") and c != "DOB_too_young"]
    young_dob_rows[young_display_cols].to_excel(writer, sheet_name="Pull out - Young DOB", index=False)
    print(f"Successfully created sheet for: Pull out - Young DOB (Rows: {len(young_dob_rows)})")


    staff_missing_summary.to_excel(writer, sheet_name="Missing Staff Summary", index=False)
    print(f"Successfully created sheet for: Missing Staff Summary (Rows: {len(staff_missing_summary)})")


    # Pull-out tab listing staff rows that need correction; red highlights are applied after writing
    staff_display_cols = [
        c for c in staff_missing_rows.columns
        if not c.endswith("_missing") and c != "_staff_id"
    ]
    staff_missing_rows[staff_display_cols].to_excel(writer, sheet_name="Pull out - Missing Staff Info", index=False)
    print(f"Successfully created sheet for: Pull out - Missing Staff Info (Rows: {len(staff_missing_rows)})")

    for site_name, final_df in site_tables.items():
        safe_sheet_name = (
            str(site_name)[:31]
            .replace(":", "").replace("/", "").replace("\\", "")
            .replace("?", "").replace("*", "")
        )
        final_df.to_excel(writer, sheet_name=safe_sheet_name, index=False)
        print(f"Successfully created sheet for: {safe_sheet_name} (Rows: {len(final_df)})")

# Apply highlights after file is fully closed
apply_missing_highlights(
    output_filename,
    total_missing_rows,
    flag_cols,
    young_dob_rows,
    staff_missing_rows=staff_missing_rows,
    staff_flag_cols=staff_flag_cols,
    site_tables=site_tables
)

print(f"\nAll data saved to '{output_filename}'! Triggering download...")
# files.download(output_filename)

import os
print(os.path.abspath(output_filename))



Successfully created sheet for: Student Summary Statistics (Rows: 4)
Successfully created sheet for: Missing Student Summary (Rows: 4)
Successfully created sheet for: Missing Site Info (Rows: 0)
Successfully created sheet for: Missing - P.S. 165 Robert E. Si (Rows: 1)
Successfully created sheet for: Missing - STEM Institute (Rows: 1)
Successfully created sheet for: Pull out - Young DOB (Rows: 1)
Successfully created sheet for: Missing Staff Summary (Rows: 4)
Successfully created sheet for: Pull out - Missing Staff Info (Rows: 22)
Successfully created sheet for: High School of Arts and Technol (Rows: 66)
Successfully created sheet for: P.S. 165 Robert E. Simon (Rows: 101)
Successfully created sheet for: STEM Institute (Rows: 24)
Highlights applied.

All data saved to 'Processed_Site_Tables.xlsx'! Triggering download...
/content/Processed_Site_Tables.xlsx
